# Enriquecimiento y Unificación

Crea las tablas de catalogo/lookup en RAW que el notebook 03 usara para construir la OBT:
- **Taxi Zones** (LocationID -> zone, borough)
- **Payment Types** (1=Credit card, 2=Cash, ...)
- **Rate Codes** (1=Standard, 2=JFK, ...)
- **Vendors** (1=CMT, 2=VeriFone)

## Imports y configuración

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import snowflake.connector

SNOWFLAKE_ACCOUNT   = os.environ["SNOWFLAKE_ACCOUNT"]
SNOWFLAKE_USER      = os.environ["SNOWFLAKE_USER"]
SNOWFLAKE_PASSWORD  = os.environ["SNOWFLAKE_PASSWORD"]
SNOWFLAKE_DATABASE  = os.environ["SNOWFLAKE_DATABASE"]
SNOWFLAKE_WAREHOUSE = os.environ["SNOWFLAKE_WAREHOUSE"]
SNOWFLAKE_ROLE      = os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN")
SCHEMA_RAW          = os.environ.get("SNOWFLAKE_SCHEMA_RAW", "RAW")

spark = SparkSession.builder \
    .appName("P3_Enriquecimiento") \
    .config("spark.jars.packages",
            "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4,"
            "net.snowflake:snowflake-jdbc:3.16.1") \
    .getOrCreate()

sf_options = {
    "sfURL": f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com",
    "sfUser": SNOWFLAKE_USER,
    "sfPassword": SNOWFLAKE_PASSWORD,
    "sfDatabase": SNOWFLAKE_DATABASE,
    "sfSchema": SCHEMA_RAW,
    "sfWarehouse": SNOWFLAKE_WAREHOUSE,
    "sfRole": SNOWFLAKE_ROLE,
}

print(f"Spark {spark.version} | {SNOWFLAKE_DATABASE}.{SCHEMA_RAW}")

## Taxi Zone Lookup

Descarga el CSV oficial de NYC TLC (~12 KB, 265 zonas) y lo sube a RAW.

In [ ]:
import urllib.request

TAXI_ZONES_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
local_csv = "/tmp/taxi_zone_lookup.csv"
urllib.request.urlretrieve(TAXI_ZONES_URL, local_csv)

df_zones = spark.read.option("header", True).option("inferSchema", True).csv(local_csv)
df_zones = df_zones.toDF(*[c.lower() for c in df_zones.columns])

df_zones.write.format("net.snowflake.spark.snowflake") \
    .options(**sf_options) \
    .option("dbtable", "TAXI_ZONES") \
    .mode("overwrite") \
    .save()

print(f"TAXI_ZONES: {df_zones.count()} filas")
df_zones.show(5)

## Catalogos: Payment Type, Rate Code, Vendor

Tablas de referencia estaticas basadas en el diccionario oficial de NYC TLC.

In [ ]:
# Payment Types
payment_data = [
    (1, "Credit card"), (2, "Cash"), (3, "No charge"),
    (4, "Dispute"), (5, "Unknown"), (6, "Voided trip")
]
df_payment = spark.createDataFrame(payment_data, ["payment_type", "payment_type_desc"])

df_payment.write.format("net.snowflake.spark.snowflake") \
    .options(**sf_options) \
    .option("dbtable", "PAYMENT_TYPES") \
    .mode("overwrite") \
    .save()

print("PAYMENT_TYPES:")
df_payment.show()

In [ ]:
# Rate Codes
rate_data = [
    (1, "Standard rate"), (2, "JFK"), (3, "Newark"),
    (4, "Nassau/Westchester"), (5, "Negotiated fare"),
    (6, "Group ride"), (99, "Unknown")
]
df_rate = spark.createDataFrame(rate_data, ["rate_code_id", "rate_code_desc"])

df_rate.write.format("net.snowflake.spark.snowflake") \
    .options(**sf_options) \
    .option("dbtable", "RATE_CODES") \
    .mode("overwrite") \
    .save()

print("RATE_CODES:")
df_rate.show()

In [ ]:
# Vendors
vendor_data = [
    (1, "Creative Mobile Technologies"), (2, "VeriFone Inc.")
]
df_vendor = spark.createDataFrame(vendor_data, ["vendor_id", "vendor_name"])

df_vendor.write.format("net.snowflake.spark.snowflake") \
    .options(**sf_options) \
    .option("dbtable", "VENDORS") \
    .mode("overwrite") \
    .save()

print("VENDORS:")
df_vendor.show()

## Verificación

Confirma que las 4 tablas de lookup existen en RAW.

In [ ]:
conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT, user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD, database=SNOWFLAKE_DATABASE,
    schema=SCHEMA_RAW, warehouse=SNOWFLAKE_WAREHOUSE, role=SNOWFLAKE_ROLE,
)
cursor = conn.cursor()

for table in ["TAXI_ZONES", "PAYMENT_TYPES", "RATE_CODES", "VENDORS"]:
    cursor.execute(f"SELECT COUNT(*) FROM {SCHEMA_RAW}.{table}")
    count = cursor.fetchone()[0]
    print(f"  {table}: {count} filas")

conn.close()
print("\nLookups listos para notebook 03.")